In [1]:
!pip install openai wikipedia python-docx

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: C:\Users\alfrancine\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import os
from docx import Document
from openai import OpenAI

# =========================
# CONNECT TO LM STUDIO
# =========================
# Make sure LM Studio is running locally on port 1234
client = OpenAI(base_url="http://127.0.0.1:1234/v1", api_key="lm-studio")

# =========================
# CREATE DOCUMENTS FOLDER
# =========================
if not os.path.exists("documents"):
    os.makedirs("documents")


# =========================
# COMMAND STATUS DISPLAY
# =========================
def show_command_status(command_type, details=""):
    if details:
        print(f"\n[{command_type}] User: {details}")


# =========================
# SAVE TEXT TO DOCX
# =========================
def save_to_doc(topic, content):
    # Standardize filename format
    safe_topic = topic.lower().replace(" ", "_").strip()
    filename = f"documents/{safe_topic}.docx"

    doc = Document()
    doc.add_heading(topic.title(), level=1)
    doc.add_paragraph(content)

    doc.save(filename)
    return filename


# =========================
# MAIN CHATBOT
# =========================
def chatbot():
    print("=================================")
    print("      AI CHATBOT WITH AGENTS     ")
    print("=================================")

    print("\nCommands:")
    print("1. save the <topic> as docs")
    print("2. exit")

    while True:
        try:
            user_input = input("\nUser: ").strip()
        except (KeyboardInterrupt, EOFError):
            print("\n\nBot: Goodbye!")
            break

        if not user_input:
            continue

        # =========================
        # EXIT COMMAND
        # =========================
        if user_input.lower() == "exit":
            show_command_status("EXIT")
            print("Bot: Goodbye!")
            break

        # =========================
        # SAVE DOC COMMAND
        # =========================
        elif "save the" in user_input.lower() and "as docs" in user_input.lower():
            try:
                show_command_status("SAVE DOCUMENT", user_input)

                # Dynamically slice out the topic regardless of surrounding words
                lower_input = user_input.lower()
                start_idx = lower_input.find("save the") + len("save the")
                end_idx = lower_input.find("as docs")

                topic = user_input[start_idx:end_idx].strip()

                if not topic:
                    print(
                        "Bot: I couldn't identify the topic. Please use the format: 'save the [topic] as docs'"
                    )
                    continue

                print(f"Bot: Generating detailed content for '{topic}'...")

                # Call LM Studio to get detailed information about the topic
                response = client.chat.completions.create(
                    model="qwen/qwen3-1.7b",
                    messages=[
                        {
                            "role": "system",
                            "content": "You are an expert researcher. Write a detailed, comprehensive summary about the user's topic.",
                        },
                        {
                            "role": "user",
                            "content": f"Write a comprehensive overview of: {topic}",
                        },
                    ],
                    temperature=0.7,
                )

                ai_content = response.choices[0].message.content

                print("Bot: Creating and writing to Word document...")
                filename = save_to_doc(topic, ai_content)

                print("Bot: Document saved successfully!")
                print(f"Bot: File location -> {filename}")

            except Exception as e:
                print("Bot: Error generating or saving document.")
                print("Details:", e)

        # =========================
        # NORMAL CHAT MODE
        # =========================
        else:
            try:
                show_command_status("CHAT MODE", user_input)

                response = client.chat.completions.create(
                    model="qwen/qwen3-1.7b",
                    messages=[
                        {
                            "role": "system",
                            "content": "You are a helpful AI assistant.",
                        },
                        {"role": "user", "content": user_input},
                    ],
                    temperature=0.7,
                )

                ai_response = response.choices[0].message.content
                print(f"\nBot: {ai_response}")

            except Exception as e:
                print("Bot: Error connecting to LM Studio.")
                print("Details:", e)


# =========================
# RUN CHATBOT
# =========================
if __name__ == "__main__":
    chatbot()

      AI CHATBOT WITH AGENTS     

Commands:
1. save the <topic> as docs
2. exit

[CHAT MODE] User: hello, I'm Alfrancine

Bot: 

Hello! Nice to meet you, Alfrancine. How are you today? 😊

[CHAT MODE] User: what is compputer engineering

Bot: 

Computer Engineering is a multidisciplinary field that combines **computer science** (theoretical and algorithmic aspects) with **electrical engineering** (hardware design and circuitry). It focuses on designing, developing, and maintaining **computing systems**, including hardware, software, and networks.

### Key Areas of Focus:
1. **Hardware Design**:  
   - Creating components like CPUs, GPUs, memory, storage, and circuits.  
   - Working with microprocessors, firmware, and embedded systems.

2. **Software Development**:  
   - Developing operating systems, programming languages, algorithms, and software tools.  
   - Optimizing code for performance and efficiency.

3. **System Architecture**:  
   - Designing how hardware and software inter